# 04A 真实 COF ML 综合案例：CO₂ adsorption

> 🟢 **Level A · 必须掌握** | 综合项目 | 完成标准：独立完成 `data → EDA → clean → features → models → CV → final test → interpretation`。

target 是公开 GCMC 数据，不是实验测量。必须保留条件、单位与 provenance。


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline


In [ ]:
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
target='CO2-1 bar (mol/kg)'
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].copy(); X['LCD_PLD_ratio']=X['LCD (Å)']/X['PLD (Å)']; X=X.replace([float('inf'), -float('inf')], float('nan')); y=df[target]
display(df.head()); display(df.describe().T)


In [ ]:
from sklearn.model_selection import KFold,cross_validate,train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,r2_score
Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=0.2,random_state=42)
cv=KFold(5,shuffle=True,random_state=42)
models={'Ridge':make_pipeline(StandardScaler(),Ridge()),'Random Forest':RandomForestRegressor(n_estimators=400,random_state=42,n_jobs=-1),'Extra Trees':ExtraTreesRegressor(n_estimators=400,random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingRegressor(random_state=42)}
models={name:make_pipeline(SimpleImputer(strategy='median'),model) for name,model in models.items()}
rows=[]
for name,m in models.items():
    s=cross_validate(m,Xdev,ydev,cv=cv,scoring={'mae':'neg_mean_absolute_error','r2':'r2'}); rows.append([name,(-s['test_mae']).mean(),(-s['test_mae']).std(),s['test_r2'].mean()])
results=pd.DataFrame(rows,columns=['Model','CV_MAE','CV_MAE_std','CV_R2']).sort_values('CV_MAE'); display(results)
best=models[results.iloc[0]['Model']].fit(Xdev,ydev); p=best.predict(Xtest)
print('test MAE =',mean_absolute_error(ytest,p),'test R² =',r2_score(ytest,p))


## 必做 sensitivity tests
1. 加入 `KCO2 (mol/kg/Pa)` 后重复；2. 换用 0.1/5/10 bar CO₂ 数据；3. 比较 feature importance 随压力变化。

### Dataset card
记录 source/DOI、structures、target、T/P/unit、descriptors、split、preprocessing、model、metrics 和版本。


## 拓展实验：第二套真实数据
完成 COFSpace 主案例后再运行。以下变量对应独立的 30 bar 任务，不能把它的 MAE 与 1 bar 案例直接比较。


## 2. Dataset B — CURATED-COFs adsorption properties

`nachatz/cof-data` 整理了来自 CURATED-COFs / Materials Cloud 的 adsorption tasks。`properties.csv` 包含 H₂、O₂、CO₂、CH₄、N₂、Xe、Kr、H₂O、H₂S 等性质；`simple_features.csv` 提供 ASA、density 和 largest sphere 等简单结构特征。两个表通过 COF ID 连接。

这一部分特别适合练习：**不同来源的 feature table 和 property table 如何通过稳定 ID merge。**


### 先审计合并
先检查空键、重复键、未匹配行和 target 单位。直接 inner join 会隐藏丢失材料。下方为整理后的派生表，引用时还应追溯 Materials Cloud 原记录；科研使用前从原记录确认温度和模拟协议。


In [ ]:
prop_url = 'https://raw.githubusercontent.com/nachatz/cof-data/main/properties.csv'
feat_url = 'https://raw.githubusercontent.com/nachatz/cof-data/main/simple_features.csv'
properties = pd.read_csv(prop_url)
simple_features = pd.read_csv(feat_url)
assert simple_features['cof'].notna().all() and properties['name'].notna().all()
assert simple_features['cof'].is_unique and properties['name'].is_unique
coverage = simple_features[['cof']].merge(properties[['name']], left_on='cof', right_on='name', how='outer', indicator=True, validate='one_to_one')
display(coverage['_merge'].value_counts().to_frame('rows'))
display(properties['co2_ads_unit'].value_counts(dropna=False).to_frame('rows'))
assert properties['co2_ads_unit'].dropna().nunique() == 1, 'Resolve mixed target units before training'
dataset_b = simple_features.merge(properties, left_on='cof', right_on='name', how='inner', validate='one_to_one')
print('features:', simple_features.shape, 'properties:', properties.shape, 'merged:', dataset_b.shape)
display(dataset_b[['cof','ASA_m^2/g','Density','LS','co2_30bar','co2_ads_unit','co2_henry','co2_henry_unit']].head())


In [ ]:
# 只用三个廉价结构特征预测 CO2 30 bar uptake：这是 deliberately simple baseline
df = dataset_b[['ASA_m^2/g','Density','LS','co2_30bar']].dropna()
X = df[['ASA_m^2/g','Density','LS']]
y = df['co2_30bar']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
m2 = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
p2 = m2.predict(X_test)
print('n =', len(df), 'MAE =', mean_absolute_error(y_test, p2), 'R2 =', r2_score(y_test, p2))


## 4. 三个数据集分别教什么？

| Dataset | 规模/类型 | 适合教学的问题 |
|---|---|---|
| COFSpace / CoRE COF | ~10³，真实模拟 label | 标准 supervised regression、feature importance、多气体/多压力 target |
| CURATED-COFs adsorption | 实验 COF + 计算性质 | CIF/ID/property merge、单位与条件、small-data baseline |
| ReDD-COFFEE HTS | ~10⁵ hypothetical COFs | fixed split、feature reduction、surrogate screening、SHAP、scale |

不要把三个数据集直接拼成一个训练表。它们的结构来源、计算协议、特征定义和 target 条件不同。正确做法是先分别建立 dataset card，再判断是否可以做 transfer learning、external validation 或 domain-shift study。


## 5. 建议练习

1. 在 COFSpace 中比较 0.1、1、5、10 bar 的 CO₂ 模型，观察 feature importance 是否随压力变化。
2. 在 CURATED 数据中分别预测 `co2_30bar` 与 `co2_henry`，解释为什么控制因素可能不同。
3. 将 04B 从 CIF 计算出的 composition descriptors 与 Dataset B 的 property table 通过 COF ID 合并。
4. 对 ReDD-COFFEE 使用作者提供的固定 train/test list，而不是重新随机切分，并比较结果。
5. 为每个实验写出 dataset card：source、structure type、target、T/P、unit、simulation method、features、split、license/citation。


## 数据来源与引用

- COFSpace: G. Onder Aksu et al., *The COF Space: Materials Features, Gas Adsorption, and Separation Performances Assessed by Machine Learning*, ACS Materials Letters. Data and scripts: https://github.com/gokhanonderaksu/COFSpace
- CURATED-COFs / Materials Cloud: D. Ongari et al., *Building a consistent and reproducible database for adsorption evaluation in Covalent-Organic Frameworks*. Data DOI: 10.24435/materialscloud:z6-jn.
- ReDD-COFFEE / CO₂ capture HTS: https://github.com/jsdvos/SupportingInformation_CO2captureHTS_2024

使用这些数据发表研究时，请引用原始论文和数据记录，而不是只引用本教程。


## 数据来源与扩展阅读
[Dataset contracts / 数据使用约定](../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
